# 07. 카메라 Exposure WhiteBalance Gamma Noise 적용

카메라 설정과 후처리가 image distribution을 어떻게 바꾸는지 확인합니다.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "synthetic_metal_utils.py").exists():
    search_roots = [Path.cwd(), *Path.cwd().parents]
    search_patterns = ["synthetic_metal_utils.py", "*/synthetic_metal_utils.py", "*/*/synthetic_metal_utils.py"]
    for root in search_roots:
        for pattern in search_patterns:
            matches = list(root.glob(pattern))
            if matches:
                NOTEBOOK_DIR = matches[0].parent
                break
        if (NOTEBOOK_DIR / "synthetic_metal_utils.py").exists():
            break

sys.path.append(str(NOTEBOOK_DIR))
DATA_ROOT = NOTEBOOK_DIR / "data" / "synthetic_metal_seg"

from synthetic_metal_utils import *
set_korean_font()
set_seed(7)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("DATA_ROOT:", DATA_ROOT)

## camera effect sweep

같은 image에 exposure, gamma, white balance, blur, noise를 적용합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)
image, mask, meta = sample_from_domain(rng, DEFAULT_DOMAIN_SPECS[0], 128, "base", 0)
settings = [
    (0.75, (1.0, 1.0, 1.0), 1.0, 0.0, 1.0),
    (1.25, (1.0, 1.0, 1.0), 1.0, 0.0, 1.0),
    (1.0, (1.15, 0.9, 0.9), 1.0, 0.0, 1.0),
    (1.0, (1.0, 1.0, 1.0), 0.65, 0.0, 1.0),
    (1.0, (1.0, 1.0, 1.0), 1.25, 0.6, 6.0),
]
fig, axes = plt.subplots(1, len(settings), figsize=(11, 2.4))
for ax, setting in zip(axes, settings):
    out = apply_camera_effects(image, rng, *setting)
    ax.imshow(out)
    ax.set_title(str(setting[:3]), fontsize=7)
    ax.axis("off")
plt.tight_layout()

## 현장 환원 포인트

카메라 exposure/gain/white balance가 자동으로 바뀌는 설비라면 domain shift가 커질 수 있습니다.

In [ ]:
['exposure', 'white_balance_r/g/b', 'gamma', 'blur_sigma', 'noise_level']